# 02 — Normalizasyon Görsel Kontrolü (K-02)

Sıra: **log fiyat → 32 noktaya resample → z-score**. Kontrol edilenler:

1. Aynı şekil, 100x farklı fiyat seviyesinde → aynı vektör
2. Farklı bar uzunlukları aynı vektör uzayında buluşuyor mu
3. ‖z‖ = √32 (Aşama 1'in Pearson korelasyonuna eşitliği buna dayanır)
4. Çizim yolu: y ters çevrilir, **log uygulanmaz**

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 200)
OUT = Path("notebooks/output"); OUT.mkdir(parents=True, exist_ok=True)
TF = "1d"

from engine.config import N_POINTS
from engine.normalize.pipeline import normalize_window, normalize_drawing
from tools.shapes import shape

sekil = shape("head_and_shoulders", 64)
ucuz = 0.002 * np.exp(0.3 * sekil)     # penny-coin
pahali = 60_000 * np.exp(0.3 * sekil)  # BTC seviyesi

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
axes[0].plot(ucuz, color="#c94f4f"); axes[0].set_title("ham fiyat — 0,002 $")
axes[1].plot(pahali, color="#2b7a78"); axes[1].set_title("ham fiyat — 60.000 $")
axes[2].plot(normalize_window(ucuz), "o-", label="ucuz", color="#c94f4f")
axes[2].plot(normalize_window(pahali), "x--", label="pahalı", color="#2b7a78")
axes[2].set_title("normalize — üst üste"); axes[2].legend()
fig.tight_layout(); fig.savefig(OUT / "normalizasyon_olcek.png", dpi=110)

fark = np.max(np.abs(normalize_window(ucuz) - normalize_window(pahali)))
print(f"maksimum fark: {fark:.2e}  (atol=1e-6 bekleniyor)")
print(f"norm: {np.linalg.norm(normalize_window(ucuz)):.6f}  = sqrt(32) = {np.sqrt(32):.6f}")

maksimum fark: 4.57e-14  (atol=1e-6 bekleniyor)
norm: 5.656854  = sqrt(32) = 5.656854


In [2]:
# Farklı bar uzunlukları aynı uzayda buluşuyor mu?
fig, ax = plt.subplots(figsize=(7, 3.4))
for L in (24, 40, 64, 100):
    v = normalize_window(100 * np.exp(0.3 * shape("cup_and_handle", L)))
    ax.plot(v, label=f"{L} bar")
ax.set_title("aynı formasyon, farklı pencere uzunlukları → 32 nokta"); ax.legend()
fig.tight_layout(); fig.savefig(OUT / "normalizasyon_uzunluk.png", dpi=110)

taban = normalize_window(100 * np.exp(0.3 * shape("cup_and_handle", 24)))
for L in (40, 64, 100):
    v = normalize_window(100 * np.exp(0.3 * shape("cup_and_handle", L)))
    print(f"24 bar ile {L:3d} bar korelasyonu: {np.corrcoef(taban, v)[0,1]:.4f}")

24 bar ile  40 bar korelasyonu: 0.9999
24 bar ile  64 bar korelasyonu: 0.9999
24 bar ile 100 bar korelasyonu: 0.9999


In [3]:
# Çizim yolu: y ters çevrilir, log UYGULANMAZ
kanvas_y = np.array([180, 150, 120, 140, 90, 60, 95, 130, 70, 40], dtype=float)  # ekran koordinatı
cizim = normalize_drawing(kanvas_y)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].plot(kanvas_y, "o-"); axes[0].invert_yaxis(); axes[0].set_title("kanvas (y aşağı artar)")
axes[1].plot(cizim, "o-", color="#2b7a78"); axes[1].set_title("normalize (fiyat mantığı)")
fig.tight_layout(); fig.savefig(OUT / "normalizasyon_cizim.png", dpi=110)
print("düz yatay çizgi reddediliyor mu?", normalize_drawing(np.full(40, 100.0)) is None)

düz yatay çizgi reddediliyor mu? True


In [4]:
# Gerçek veriden bir pencere: ham vs normalize
from engine.ingest.store import read_klines
df = read_klines("BTCUSDT", TF)
p = df.close.to_numpy()[-100:]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot(p); axes[0].set_title("BTCUSDT son 100 gün — ham")
axes[1].plot(normalize_window(p), color="#2b7a78"); axes[1].set_title("normalize (32 nokta)")
fig.tight_layout(); fig.savefig(OUT / "normalizasyon_gercek.png", dpi=110)